# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** — Segmentação do mercado de veículos usados de Fortaleza

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu a base **como ela
chega**: 2.565 anúncios, 23 colunas brutas, nenhuma ausência estrutural (todo vazio
é real), 62 anúncios com falha total de captura, um
valor de quilometragem claramente inválido, 61 anúncios com hodômetro incompatível com a idade do veículo, uma cauda de desconto vs. FIPE que chega a −1.249% e um único dia de coleta. Este notebook é o
passo seguinte: **aplicar as correções** e gravar a base curada que as fases
posteriores consomem.

A regra deste notebook é uma só:

> **Nenhum dado é alterado em silêncio.** Cada ajuste declara o *motivo*, mostra a
> *evidência numérica* que o justifica e registra *quantos registros* foram tocados. O
> conjunto vai para `reports/tables/ajustes-dados/registro-de-ajustes.csv`.

## Duas decisões de tratamento de ausência

> **Não há ausência estrutural.** Todo carro tem marca, câmbio e combustível — o
> que falta é real, seja porque o vendedor não informou (`aceita_troca`,
> `unico_dono`), seja porque o scraper não capturou o campo. As duas naturezas pedem
> tratamentos diferentes, mas nenhuma delas vira rótulo de "não existe".
>
> **Não há um limiar único de ausência que descarta coluna.** Um corte único de, por
> exemplo, 5% descartaria `aceita_troca` (24,7%) e
> `unico_dono` (16,1%) — duas variáveis que o Canvas do Problema pede. Em vez de um
> corte único, cada natureza de ausência recebe o tratamento que ela pede: falha de
> captura vira remoção de linha (o dado não existe em lugar nenhum); ausência opcional
> vira categoria explícita `NaoInformado` (nenhum valor é inventado).

## Ordem das operações

| # | Bloco | Por que vem nesta posição |
|:--|:---|:---|
| 1 | Leitura e mesclagem da FIPE | Reproduz a decisão do notebook 01 — é a mesma base de partida |
| 2 | Duplicidade | Anúncio repetido pesa duas vezes em qualquer estatística usada adiante |
| 3 | Ausência — falha de captura | O bloco `dataLayer` falha por completo em alguns anúncios; sem ele não há o que caracterizar |
| 4 | Inconsistências | Corrigir o valor de `km` inválido antes de qualquer estatística sobre ele |
| 5 | Ausência opcional | O que sobra depois do bloco 3 é ausência real e baixa — vira categoria, não remoção |
| 6 | Atributos derivados | Dependem das colunas já limpas |
| 7 | Hodômetro implausível | Precisa de `idade_veiculo`, derivada no bloco 6 |
| 8 | Desconto fora de faixa | Precisa de `desconto_fipe_pct`, derivada no bloco 6 |
| 9 | Valores extremos | Rodam sobre `ano`/`km` já derivados e depurados |
| 10 | Colunas sem informação | Roda por último — a concentração muda com os cortes anteriores |

## Saída

* `data/processed/anuncios_curados.parquet` — a base curada, em unidades originais;
* `data/processed/manifesto_curadoria.json` — o que entrou, o que saiu e quantos ajustes;
* `reports/tables/ajustes-dados/` — o registro de ajustes e as tabelas de evidência.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.data import ingestao, preparacao, qualidade
from src.eda import estatistica as est
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

SECAO = "ajustes-dados"
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL


def salvar(fig, nome, dados=None):
    return io.salvar_figura(fig, SECAO, nome, dados)


print(f"Projeto em {RAIZ}")

## O registro de ajustes

Toda alteração passa por `registrar`. A função não altera dado nenhum — ela só anota o
que foi feito, por quê e sobre quantos registros.

| `acao` | Significado |
|:---|:---|
| `corrigido` | O valor mudou |
| `imputado` | A célula estava vazia e passou a ter valor (ou categoria explícita) |
| `removido` | O registro (ou coluna) saiu da base |
| `verificado, sem ação` | O padrão foi conferido e a decisão foi **não** mexer |

In [ ]:
AJUSTES: list[dict] = []


def pt(numero) -> str:
    '''Formata o número no padrão brasileiro: ponto como separador de milhar.'''
    return f"{numero:,}".replace(",", ".")


def registrar(
    bloco: str,
    ajuste: str,
    motivo: str,
    afetados: int,
    unidade: str = "registros",
    acao: str = "corrigido",
):
    '''Anota um ajuste no registro, com motivo, desfecho e volume afetado.'''
    AJUSTES.append(
        {
            "n": len(AJUSTES) + 1,
            "bloco": bloco,
            "ajuste": ajuste,
            "motivo": motivo,
            "afetados": int(afetados),
            "unidade": unidade,
            "acao": acao,
        }
    )
    rotulo = unidade if int(afetados) != 1 else unidade.rstrip("s")
    print(f"{len(AJUSTES):>2}. [{acao:>20}] {ajuste} — {pt(int(afetados))} {rotulo}")

# 1 · Leitura — carregando e mesclando a FIPE

> A camada bruta em `data/raw/` é **imutável**: nada neste notebook escreve nela.

A leitura reproduz a decisão do notebook 01: mesclar `valor_fipe_olx` (a FIPE que a
própria OLX mostra) com o resultado da API pública (`fipe_fallback.csv`) para os
anúncios em que a OLX não trouxe o valor. `src/data/ingestao.carregar_bruto()`
encapsula essa lógica, para que este notebook e `preparacao.construir_base_curada()`
produzam exatamente a mesma base de partida.

In [ ]:
bruto = ingestao.carregar_bruto()

LINHAS_ENTRADA, COLUNAS_ENTRADA = bruto.shape
print(f"{pt(LINHAS_ENTRADA)} linhas x {COLUNAS_ENTRADA} colunas")
print(f"Células vazias na entrada: {pt(int(bruto.isna().sum().sum()))}")

cobertura = bruto["fonte_fipe"].value_counts()
imputados_api = int((bruto["fonte_fipe"] == "API pública (fallback)").sum())
registrar(
    "1 · Leitura",
    "Mesclar `valor_fipe_final` com o resultado da API pública da FIPE",
    "A OLX só traz FIPE em 90,7% dos anúncios (`valor_fipe_olx`). O resto foi "
    "resolvido à parte, casando marca/modelo/ano por similaridade de texto contra "
    "a API pública — 164 de 238 tentativas deram certo.",
    imputados_api,
    unidade="células",
    acao="imputado",
)

# Sobra dados que a leitura sozinha resolve: e mesmo o fallback pela API não
# encontra FIPE para todos. Fica registrado, não é uma célula corrigida.
sem_fipe = int((bruto["fonte_fipe"] == "não encontrada").sum())
print(f"Sem FIPE mesmo após o fallback: {pt(sem_fipe)} ({100*sem_fipe/len(bruto):.1f}%) — fica nulo, é reservada")

cobertura

# 2 · Duplicidade — o mesmo anúncio contado duas vezes?

A checagem revela dois fenômenos **diferentes**, e a distinção
importa:

1. **duplicidade real** — o mesmo `link` aparece duas vezes na base. Não são dois
   anúncios: é o mesmo anúncio coletado duas vezes, resultado de uma falha pontual na
   dedupe incremental da coleta (o processo foi pausado e retomado várias vezes por
   queda de energia/internet, ver `Software/coleta/HISTORICO.md`);
2. **duplicidade coincidente no espaço de atributos** — depois de remover a real,
   ainda sobram grupos de anúncios com `link` distinto mas as mesmas 13 características
   analisadas (marca, câmbio, combustível, carroceria, portas, tipo de vendedor,
   aceita_troca, único_dono, município, ano, km, preço, bairro). São anúncios
   genuinamente diferentes que coincidem por serem carros comuns (ex.: dois "Volkswagen
   Gol 1.0 Manual Flex Hatch, Fortaleza, 2018" de vendedores diferentes).

**Decisão:** remove-se a primeira, mantém-se a segunda — o `link` distinto é a prova de
que são anúncios diferentes, e para a segmentação cada anúncio é uma observação de
mercado válida, ainda que dois carros comuns ocupem o mesmo ponto do espaço.

In [ ]:
links_duplicados = bruto[bruto.duplicated("link", keep=False)].sort_values("link")
io.salvar_tabela(links_duplicados, SECAO, "links-duplicados-antes", indice=False)

antes_dedupe = len(bruto)
dados = bruto.drop_duplicates(subset="link", keep="first").copy()

registrar(
    "2 · Duplicidade",
    "Remover anúncios com `link` repetido",
    "5 links aparecem duas vezes na base bruta — falha pontual na dedupe "
    "incremental da coleta, não dois anúncios distintos.",
    antes_dedupe - len(dados),
    unidade="registros",
    acao="removido",
)

links_duplicados[["titulo", "link"]]

# 3 · Ausência — falha de captura do scraper

As sete colunas do bloco `window.dataLayer` (`marca`, `câmbio`, `combustível`,
`carroceria`, `portas`, `tipo de vendedor`, `município`) vêm **juntas** da mesma
chamada à página do anúncio. Quando o bloco falha ao carregar, as sete ficam vazias ao
mesmo tempo — é falha de captura, não ausência de informação do anúncio.

O gráfico separa essa falha total (`marca` sempre presente quando qualquer uma das
sete está) da ausência parcial, que é tratada à parte no bloco 5.

In [ ]:
NUCLEO = dic.NUCLEO_DATALAYER

ausentes = (
    dados.isna()
    .sum()
    .rename("ausentes")
    .to_frame()
    .assign(percentual=lambda t: (100 * t.ausentes / len(dados)).round(2))
    .query("ausentes > 0")
    .sort_values("ausentes", ascending=False)
    .reset_index(names="variavel")
)
ausentes["natureza"] = np.select(
    [
        ausentes.variavel.isin(NUCLEO),
        ausentes.variavel.isin(dic.AUSENCIA_OPCIONAL),
        ausentes.variavel == dic.RESERVADA_FIPE,
    ],
    ["falha de captura (dataLayer)", "opcional (não informado)", "reservada (FIPE)"],
    default="outra",
)

CORES_NATUREZA = {
    "falha de captura (dataLayer)": ALERTA,
    "opcional (não informado)": DESTAQUE,
    "reservada (FIPE)": config.CINZA,
    "outra": REFERENCIA,
}

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(
    data=ausentes, x="variavel", y="percentual", hue="natureza", palette=CORES_NATUREZA, ax=ax
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.1f%%", fontsize=8, padding=2)
ax.tick_params(axis="x", rotation=60)
ax.set_xlabel("Colunas com ao menos uma célula vazia")
ax.set_ylabel("% de células ausentes")
ax.set_title(f"Natureza da ausência antes dos ajustes — {len(ausentes)} colunas afetadas")
ax.legend(title="Natureza")

salvar(fig, "ausencia-por-natureza", ausentes)
ausentes

In [ ]:
falha_total = dados[NUCLEO].isna().all(axis=1)
evidencia_falha = dados.loc[falha_total, ["titulo", "preco", "km", "ano"] + NUCLEO]
io.salvar_tabela(evidencia_falha, SECAO, "falha-total-dataLayer", indice=False)

antes_falha = len(dados)
dados = dados[~falha_total].copy()

registrar(
    "3 · Ausência",
    "Remover anúncios com falha total do bloco `dataLayer`",
    "As sete colunas do núcleo ficam vazias juntas — falha de captura do scraper. "
    "Sem elas não há como caracterizar o veículo (só sobram título/preço/km/ano, "
    "que vêm de outra etapa da coleta).",
    antes_falha - len(dados),
    unidade="registros",
    acao="removido",
)

antes_ano = len(dados)
dados = dados[dados["ano"].notna()].copy()
registrar(
    "3 · Ausência",
    "Remover o anúncio sem `ano`",
    "Um único registro. `ano` é a variável mais central da caracterização e não "
    "tem substituto plausível.",
    antes_ano - len(dados),
    unidade="registros",
    acao="removido",
)

print(f"Células vazias restantes: {pt(int(dados.isna().sum().sum()))}")
evidencia_falha.head()

# 4 · Inconsistências — o valor de `km` inválido

A EDA (notebook 01, insight 1) já apontou: um único registro traz `km = 999.998` — bem
acima de qualquer outro valor da base e suspeitosamente próximo de 10⁶ − 2, padrão
típico de valor de preenchimento de sistema, não de uma quilometragem real. Como não
há como inferir o valor verdadeiro a partir de outra coluna, o registro é removido.

Abaixo, também rodam as checagens formais de plausibilidade do domínio
(`src/data/qualidade.py`), como registro do que foi verificado — sem uma autoridade
externa citável para justificar remoção
automática, elas ficam **registradas**, não removidas.

In [ ]:
antes_km = len(dados)
maior_km = dados.loc[dados["km"].idxmax(), ["titulo", "ano", "km", "preco"]]
dados = dados[dados["km"] != preparacao.KM_SENTINELA].copy()

registrar(
    "4 · Inconsistências",
    f"Remover o anúncio com `km` = {pt(preparacao.KM_SENTINELA)}",
    "Valor isolado, muito acima do segundo maior (800.000) e sugestivamente "
    "próximo de 10⁶ − 2 — padrão de valor de preenchimento, não de uma "
    "quilometragem real.",
    antes_km - len(dados),
    unidade="registros",
    acao="removido",
)
print(maior_km.to_frame("removido"))

# Checagens de plausibilidade — precisam de `idade_veiculo`, calculada aqui só
# para a checagem (a derivação "oficial" acontece no bloco 6).
dados["idade_veiculo"] = (config.ANO_REFERENCIA - dados["ano"]).clip(lower=0)
plausibilidade_antes = qualidade.valores_implausiveis(dados)
io.salvar_tabela(plausibilidade_antes.reset_index(), SECAO, "plausibilidade-verificacao", indice=False)

for regra, linha in plausibilidade_antes.iterrows():
    if linha.situacao == "verificada":
        registrar(
            "4 · Inconsistências",
            f"Verificar: {regra}",
            "Checagem de plausibilidade do domínio, sem autoridade externa para "
            "justificar remoção automática — fica registrada para leitura humana.",
            int(linha.registros),
            unidade="registros",
            acao="verificado, sem ação",
        )

plausibilidade_antes

# 5 · Ausência opcional — a categoria `NaoInformado`

O que sobra depois do bloco 3 é ausência **real e baixa**: `câmbio`, `combustível`,
`carroceria` e `portas` (abaixo de 7% cada, ausência parcial no próprio anúncio, não
falha de captura) e `aceita_troca`/`único_dono` (até 25%, dependem do vendedor
informar). Nenhuma delas passa por imputação estatística — nenhum valor numérico ou
categórico é inventado. Cada uma recebe a categoria explícita `NaoInformado`, que
preserva a informação de que o dado falta, sem descartar a linha inteira.

Isso é uma escolha deliberada: um limiar único de ausência (por exemplo, >5% →
descarta coluna) descartaria `aceita_troca` (24,7%) e `único_dono` (16,1%) — duas
variáveis que o
Canvas do Problema pede, e descartá-las jogaria fora informação de negócio.

In [ ]:
ANTES = {c: int(dados[c].isna().sum()) for c in dic.AUSENCIA_OPCIONAL if c in dados.columns}

dados = preparacao._rotular_ausencia_opcional(dados)

rotulagem = pd.DataFrame(
    [
        {
            "coluna": coluna,
            "celulas_rotuladas": celulas,
            "pct_da_coluna": round(100 * celulas / len(dados), 2),
        }
        for coluna, celulas in ANTES.items()
        if celulas
    ]
).sort_values("celulas_rotuladas", ascending=False)
io.salvar_tabela(rotulagem, SECAO, "rotulagem-ausencia-opcional", indice=False)

registrar(
    "5 · Ausência opcional",
    "Rotular como `NaoInformado` a ausência real nas colunas opcionais",
    "Ausência legítima e baixa (até 24,7%): vira categoria explícita, sem inventar "
    "valor. `aceita_troca` e `unico_dono` continuam no espaço de atributos.",
    int(rotulagem.celulas_rotuladas.sum()),
    unidade="células",
    acao="imputado",
)

print(f"Células vazias restantes (fora da FIPE reservada): "
      f"{pt(int(dados.drop(columns=[dic.RESERVADA_FIPE, 'valor_fipe_olx']).isna().sum().sum()))}")
rotulagem

# 6 · Atributos derivados

Três colunas novas, construídas a partir das que já estão limpas:

* **`idade_veiculo`** — `config.ANO_REFERENCIA` (2026, o ano da coleta) menos `ano`.
  Estatisticamente equivalente a `ano` para clusterização (é uma transformação afim:
  padronizar uma ou outra dá a mesma geometria), mas lê melhor em relatório
  ("idade média de 8 anos" em vez de "ano médio 2018");
* **`km_por_ano`** — quilometragem dividida pela idade (piso em 1 ano, para não
  dividir por zero em carro do ano). Mede **intensidade de uso**, informação que `km`
  e `ano` sozinhos não capturam: dois carros de 2018 com 80.000 km têm o mesmo `km`,
  mas se um tem `idade_veiculo` = 8 e outro = 2 (ano de fabricação diferente do ano de
  referência do modelo), a intensidade de uso é muito diferente;
* **`zero_km`** — flag binária (`km == 0`). Entra no espaço de atributos
  (`dic.BINARIAS`) porque "km zero" é uma categoria de negócio distinta ("carro de
  vitrine"), não só a ponta inferior da distribuição de `km`.

`desconto_fipe_pct` deriva das **reservadas** (`preco`, `valor_fipe_final`) e por isso
não entra em nenhuma etapa de modelagem — fica na base curada apenas para a avaliação
*a posteriori* (Pergunta 5 do projeto: quais segmentos representam oportunidade).

In [ ]:
dados = preparacao._derivar(dados)

DERIVADAS = ["idade_veiculo", "km_por_ano", "zero_km", "desconto_fipe_pct"]

registrar(
    "6 · Derivadas",
    "Derivar `idade_veiculo`, `km_por_ano` e `zero_km` (espaço de atributos) "
    "mais `desconto_fipe_pct`, reservada",
    "Idade na data da coleta lê melhor que o ano absoluto; km/ano mede intensidade "
    "de uso, informação que km e ano sozinhos não capturam.",
    len(DERIVADAS),
    unidade="colunas",
)

resumo_derivadas = dados[DERIVADAS].describe().T.round(2)
io.salvar_tabela(resumo_derivadas.reset_index(names="variavel"), SECAO, "atributos-derivados", indice=False)
resumo_derivadas

# 7 · Hodômetro implausível — km baixa em carro velho

Há 61 anúncios que declaram **menos de 1.000 km** em veículos com **mais de 5 anos**.
Não é uma frota preservada: a mediana do grupo dá **14 km rodados por ano**, e as
linhas mostram o que realmente são — uma Belina 1982 com 100 km, um Corsa 1999 com
267 km, um Civic 2007 com 1.000 km anunciado a R$ 10 mil contra FIPE de R$ 43 mil.

O padrão é **quilometragem digitada em milhares** ("120" no lugar de 120.000) ou campo
não preenchido pelo anunciante. Isso importa muito mais do que 61 linhas sugerem: sem
a regra, esses registros se agrupam entre si — km quase zero, idade alta — e a
clusterização os promove a **um segmento inteiro**, com leitura comercial e tudo.
Um artefato de captura vira persona de negócio.

O corte de idade é o que protege o dado legítimo: zero-km e seminovos de vitrine
continuam na base, marcados por `zero_km`. A regra alcança só a combinação
impossível: rodado no tempo, parado no hodômetro.


In [ ]:
antes_hodometro = len(dados)

implausivel = (dados["km"] < preparacao.KM_MINIMA_PLAUSIVEL) & (
    dados["idade_veiculo"] > preparacao.IDADE_MINIMA_PARA_KM
)

amostra_implausivel = (
    dados.loc[implausivel, ["marca", "modelo", "ano", "km", "preco", "valor_fipe_final"]]
    .sort_values("ano")
    .head(10)
)
io.salvar_tabela(amostra_implausivel, SECAO, "hodometro-implausivel", indice=False)

dados = dados[~implausivel].copy()

registrar(
    "7 · Hodômetro",
    f"Remover km < {pt(preparacao.KM_MINIMA_PLAUSIVEL)} em veículos com mais de "
    f"{preparacao.IDADE_MINIMA_PARA_KM} anos",
    "Km digitada em milhares ou campo não preenchido: o grupo tem mediana de 14 km "
    "por ano. Sem a regra vira um segmento inteiro na clusterização.",
    antes_hodometro - len(dados),
    unidade="registros",
    acao="removido",
)

amostra_implausivel


# 8 · Desconto fora de faixa — quando a FIPE erra o carro

`desconto_fipe_pct` é `(fipe − preço) / fipe`, uma razão **sem limite inferior**: se a
FIPE de referência for baixa e o preço alto, o número dispara. E a FIPE nem sempre é
confiável — 6,4% dos anúncios têm o valor vindo de casamento por *similaridade de
texto* contra a API pública (ver a página *Fonte dos dados*), que às
vezes casa o modelo errado.

O resultado na base bruta: desconto mínimo de **−1.249%**. Um único anúncio mal casado
puxa a média de um segmento inteiro. Como a leitura de oportunidade comercial do
projeto (Pergunta 5) é construída sobre essa coluna, a cauda precisa sair **antes** de
qualquer agregação.

Diferente das regras anteriores, aqui **o anúncio fica e só o valor sai**.
`desconto_fipe_pct` é reservada — não entra no espaço de atributos —, então o defeito
está na referência de preço, não nas características que agrupam o veículo. Remover a
linha custaria um carro na segmentação para consertar um número que a segmentação nem
usa.


In [ ]:
antes_desconto = int(dados["desconto_fipe_pct"].notna().sum())

fora_da_faixa = dados["desconto_fipe_pct"].abs() > preparacao.DESCONTO_MAXIMO_PLAUSIVEL

faixa = pd.DataFrame(
    [
        {
            "momento": "antes de aparar",
            "com_leitura": antes_desconto,
            "minimo": round(dados["desconto_fipe_pct"].min(), 1),
            "maximo": round(dados["desconto_fipe_pct"].max(), 1),
            "media": round(dados["desconto_fipe_pct"].mean(), 2),
            "mediana": round(dados["desconto_fipe_pct"].median(), 2),
        }
    ]
)

dados.loc[fora_da_faixa, "desconto_fipe_pct"] = np.nan

faixa = pd.concat(
    [
        faixa,
        pd.DataFrame(
            [
                {
                    "momento": "depois de aparar",
                    "com_leitura": int(dados["desconto_fipe_pct"].notna().sum()),
                    "minimo": round(dados["desconto_fipe_pct"].min(), 1),
                    "maximo": round(dados["desconto_fipe_pct"].max(), 1),
                    "media": round(dados["desconto_fipe_pct"].mean(), 2),
                    "mediana": round(dados["desconto_fipe_pct"].median(), 2),
                }
            ]
        ),
    ],
    ignore_index=True,
)
io.salvar_tabela(faixa, SECAO, "desconto-fora-de-faixa", indice=False)

registrar(
    "8 · Desconto",
    f"Anular desconto_fipe_pct fora de +/-{preparacao.DESCONTO_MAXIMO_PLAUSIVEL:.0f}%",
    "Razão sem limite inferior alimentada por casamento de FIPE por similaridade de "
    "texto: o mínimo bruto é −1.249%. Só o valor sai; o anúncio permanece na base.",
    int(fora_da_faixa.sum()),
    unidade="leituras",
    acao="anulado",
)

faixa


# 9 · Valores extremos — 3×IQR, não 1,5×IQR

O critério de Tukey (1,5×IQR) marca 3,5% de `preco`/`valor_fipe_final` e 2,4% de `km` —
mas aplicá-lo à variável de segmentação removeria justamente os carros antigos e de
alta quilometragem que a segmentação existe para isolar como um grupo à parte, não como
ruído.

O corte usa **3×IQR**, aplicado só a `ano` e `km` (as numéricas do espaço de
atributos — `preco` e `valor_fipe_final` são reservadas e não participam deste corte).
A tabela compara os dois critérios, para que a escolha seja auditável.

In [ ]:
linhas = []
fora_da_cerca = pd.Series(False, index=dados.index)
for coluna in dic.NUMERICAS:
    serie = dados[coluna]
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    tukey = est.outliers_iqr(serie)
    extremo = (serie < q1 - preparacao.FATOR_EXTREMO * iqr) | (serie > q3 + preparacao.FATOR_EXTREMO * iqr)
    fora_da_cerca |= extremo
    linhas.append(
        {
            "variavel": coluna,
            "tukey_1_5": int(tukey.sum()),
            "pct_tukey": round(100 * tukey.mean(), 2),
            "extremos_3_0": int(extremo.sum()),
            "pct_extremos": round(100 * extremo.mean(), 2),
        }
    )

discrepantes = pd.DataFrame(linhas)
comparacao = discrepantes.melt(
    id_vars="variavel", value_vars=["pct_tukey", "pct_extremos"], var_name="criterio", value_name="percentual"
).replace({"pct_tukey": "Tukey - 1,5xIQR", "pct_extremos": "Extremo - 3xIQR"})

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=comparacao, x="percentual", y="variavel", hue="criterio",
    palette={"Tukey - 1,5xIQR": config.CINZA, "Extremo - 3xIQR": ALERTA}, ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f%%", fontsize=8, padding=2)
ax.set_xlabel("% de registros fora da cerca")
ax.set_ylabel("")
ax.set_title("Dois critérios de discrepância em ano/km — só o de 3xIQR é aplicado")
ax.margins(x=0.16)

salvar(fig, "discrepantes-dois-criterios", discrepantes)

antes_extremos = len(dados)
dados = dados[~fora_da_cerca].copy()

registrar(
    "9 · Extremos",
    f"Remover valores além de {preparacao.FATOR_EXTREMO:.0f}xIQR em ano/km",
    "A cerca de Tukey (1,5xIQR) alcançaria carros antigos e de alta quilometragem "
    "genuínos — o segmento que a clusterização precisa isolar, não descartar.",
    antes_extremos - len(dados),
    unidade="registros",
    acao="removido",
)
discrepantes

# 10 · Colunas sem informação

Coluna em que 99% dos anúncios dizem a mesma coisa não separa ninguém. Roda por
último, porque a concentração muda com os cortes anteriores.

In [ ]:
concentracao = []
for coluna in dados.columns:
    preenchida = dados[coluna].dropna()
    if not len(preenchida):
        continue
    frequencia = preenchida.value_counts(normalize=True)
    concentracao.append(
        {
            "coluna": coluna,
            "distintos": int(preenchida.nunique()),
            "valor_dominante": frequencia.index[0],
            "pct_dominante": round(100 * frequencia.iloc[0], 2),
            "protegida": coluna in preparacao.COLUNAS_PROTEGIDAS,
        }
    )

concentracao = pd.DataFrame(concentracao).sort_values("pct_dominante", ascending=False)
sem_informacao = concentracao.query("pct_dominante >= @preparacao.LIMIAR_CONCENTRACAO")
io.salvar_tabela(sem_informacao, SECAO, "colunas-sem-informacao", indice=False)

descartar = sem_informacao.query("not protegida").coluna.tolist()
dados = dados.drop(columns=descartar)
dados = dados.reset_index(drop=True)

registrar(
    "10 · Colunas",
    "Descartar as colunas concentradas acima de 99% num único valor",
    "Não separam ninguém e entram como ruído com o mesmo peso de uma variável que "
    "discrimina.",
    len(descartar),
    unidade="colunas",
    acao="removido",
)

print(f"Descartadas: {', '.join(descartar) or '(nenhuma)'}")
sem_informacao.head(10)

# Conferência da base curada

Duas verificações antes de gravar:

1. **Nulo fora da FIPE reservada** — nenhuma célula vazia deve sobrar nas colunas do
   espaço de atributos e das demais reservadas de negócio; `valor_fipe_final`
   continua com ~2,5% de nulo (declarado, não é bug — é o resíduo que nem a API
   pública resolveu);
2. **Assimetria** — confere `dic.ASSIMETRICAS` contra a medição na base curada.

In [ ]:
nulos_espaco = int(dados[dic.ESPACO_MISTO].isna().sum().sum())
print(f"Células vazias no espaço de atributos: {nulos_espaco}")
assert nulos_espaco == 0, "o espaço de atributos não pode ter célula vazia"

nulos_fipe = int(dados[dic.RESERVADA_FIPE].isna().sum())
print(f"Células vazias em {dic.RESERVADA_FIPE} (reservada, esperado > 0): {nulos_fipe} "
      f"({100*nulos_fipe/len(dados):.1f}%)")

assimetria = (
    dados[dic.NUMERICAS]
    .skew()
    .rename("assimetria")
    .to_frame()
    .assign(acima_do_limiar=lambda t: t.assimetria.abs() >= dic.LIMIAR_ASSIMETRIA)
    .reset_index(names="variavel")
)

fig, ax = plt.subplots(figsize=(10, 3.5))
sns.barplot(
    data=assimetria, x="assimetria", y="variavel", hue="acima_do_limiar",
    palette={True: ALERTA, False: BARRA}, dodge=False, ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f", fontsize=9, padding=3)
ax.axvline(dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.axvline(-dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.set_title(f"Assimetria de ano/km na base curada - limiar +-{dic.LIMIAR_ASSIMETRIA}")
ax.set_xlabel("Assimetria")
ax.set_ylabel("")
ax.legend(title=f"|assimetria| >= {dic.LIMIAR_ASSIMETRIA}")

salvar(fig, "assimetria-apos-ajustes", assimetria)

print(
    "`km` tem assimetria positiva (cauda à direita, log1p ajuda) e por isso está em "
    "`dic.ASSIMETRICAS`. `ano` passa do limiar em módulo mas com sinal negativo "
    "(cauda à esquerda) — log1p não corrige essa direção, então fica de fora."
)
assimetria

## Equivalência com o pipeline

A base curada tem **dois produtores**: este notebook, que narra cada decisão, e
`src/data/preparacao.construir_base_curada()`, que roda sem supervisão. Os dois
precisam produzir a mesma tabela.

In [ ]:
referencia = preparacao.construir_base_curada(salvar=False)

print(f"notebook: {dados.shape}   modulo: {referencia.shape}")
assert sorted(dados.columns) == sorted(referencia.columns), "conjunto de colunas difere"

linhas = []
for coluna in dados.columns:
    esquerda = dados[coluna].reset_index(drop=True)
    direita = referencia[coluna].reset_index(drop=True)
    divergem = ~((esquerda == direita) | (esquerda.isna() & direita.isna()))
    if divergem.any():
        linhas.append({"coluna": coluna, "celulas_divergentes": int(divergem.sum())})

divergencias = pd.DataFrame(linhas, columns=["coluna", "celulas_divergentes"])
io.salvar_tabela(divergencias, SECAO, "equivalencia-com-o-pipeline", indice=False)

if divergencias.empty:
    print("Sem divergência: o notebook e o pipeline produzem a mesma base curada.")
divergencias

# A base curada

* **`data/processed/anuncios_curados.parquet`** — a base curada, em unidades
  originais;
* **`data/processed/manifesto_curadoria.json`** — o que entrou, o que saiu e quantos
  ajustes;
* **`reports/tables/ajustes-dados/registro-de-ajustes.csv`** — o registro completo.

O parquet é gravado **sem transformação de escala** — `log1p` e padronização
dependem do algoritmo e vivem no pré-processador (`preparacao.construir_preprocessador`),
ajustado dentro de cada etapa de modelagem, para não vazar informação entre etapas.

In [ ]:
config.garantir_diretorios()
dados.to_parquet(config.BASE_CURADA, index=False)

registro = pd.DataFrame(AJUSTES)
io.salvar_tabela(registro, SECAO, "registro-de-ajustes", indice=False)

manifesto = {
    "origem": config.ARQUIVO_BRUTO.relative_to(RAIZ).as_posix(),
    "destino": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
    "linhas_entrada": int(LINHAS_ENTRADA),
    "colunas_entrada": int(COLUNAS_ENTRADA),
    "linhas_saida": int(len(dados)),
    "colunas_saida": int(dados.shape[1]),
    "registros_removidos": int(LINHAS_ENTRADA - len(dados)),
    "pct_registros_removidos": round(100 * (LINHAS_ENTRADA - len(dados)) / LINHAS_ENTRADA, 2),
    "fator_extremo": preparacao.FATOR_EXTREMO,
    "colunas_derivadas": DERIVADAS,
    "ajustes": int(len(registro)),
    "ajustes_por_acao": registro.acao.value_counts().to_dict(),
    "equivalente_ao_pipeline": bool(divergencias.empty),
}
io.salvar_json(manifesto, config.DATA_PROCESSED / "manifesto_curadoria.json")

print(f"Base curada gravada em {config.BASE_CURADA.relative_to(RAIZ)}")
print(f"{pt(len(dados))} registros x {dados.shape[1]} colunas - {len(registro)} ajustes registrados")
pd.Series(manifesto).to_frame("manifesto de curadoria")

In [ ]:
# O funil: de onde saiu cada registro que nao chegou a base curada.
removidos = registro.query("acao == 'removido' and unidade == 'registros'")
etapas = [{"etapa": "Base bruta", "registros": LINHAS_ENTRADA}]
restante = LINHAS_ENTRADA
for _, linha in removidos.iterrows():
    restante -= linha.afetados
    etapas.append({"etapa": linha.ajuste, "registros": restante})
funil = pd.DataFrame(etapas)
funil["removidos"] = funil.registros.shift(1).sub(funil.registros).fillna(0).astype(int)
funil["pct_do_total"] = (100 * funil.registros / LINHAS_ENTRADA).round(1)

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(data=funil, x="registros", y="etapa", color=BARRA, ax=ax)
for grupo in ax.containers:
    ax.bar_label(
        grupo,
        labels=[
            f"{pt(int(r))} ({p:.1f}%)" + (f" -{pt(int(d))}" if d else "")
            for r, p, d in zip(funil.registros, funil.pct_do_total, funil.removidos)
        ],
        fontsize=9, padding=4,
    )
ax.set_xlabel("Registros restantes")
ax.set_ylabel("")
ax.set_title(f"Funil da curadoria - {pt(LINHAS_ENTRADA)} para {pt(len(dados))} anuncios")
ax.margins(x=0.24)

salvar(fig, "funil-da-curadoria", funil)
funil

In [ ]:
registro

In [ ]:
# Amostra do que a proxima fase recebe.
dados[dic.ESPACO_MISTO + [dic.ROTULO_RESERVADO, dic.RESERVADA_VALIDACAO, dic.RESERVADA_FIPE]].head(10)

# Próximo passo

A base curada está gravada e é idêntica à que o pipeline produz. A partir daqui:

* **[`03-preparacao`](03-preparacao.ipynb)** recorta a matriz de modelagem
  (`src/data/preparacao.construir_matriz_modelagem`), define o espaço de atributos
  final e escolhe o escalonamento;
* **[`04-modelagem`](04-modelagem.ipynb)** entra na fase 4 do CRISP-DM: escolha de
  *k*, K-Means, hierárquico e/ou DBSCAN.